# 16 — Feature distributions

Split-violin plots of every MD feature, actives vs decoys, one panel per target. Answers the eyeball question: does any feature visibly separate binders from non-binders on any target? NB 21 formalises this with a z-difference, NB 28 tries a univariate ranker.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/16_feature_distributions_figK.png`.)_


> **Reader guide.** *Experiment A3:* feature-distribution comparison actives vs measured
> non-binders.
>
> **Method:** per-target split-violin plots for each MD feature; no ranking-metric yet, only
> distributions.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` + reference metadata.

In [ ]:
# --- notebook preamble ---
NB_STEM = "32_feature_distributions"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 4. Feature distributions — actives vs decoys split

The point of a feature is discrimination. For every candidate metric we plot the per-target distribution **split by activity label** (side-by-side boxes: NAVY = decoys, GOLD = actives). Individual complexes are overlaid as jittered dots.

A useful feature shifts the GOLD box away from the NAVY box **within** a target. Features whose boxes overlap in every target carry no per-target signal.


In [ ]:

SHOW_FEATURES = [
    'lig_drift_mean_A', 'lig_escape_frac',
    'lig_buried_sasa_mean_A2', 'vdw_contacts_mean',
    'n_hb_mean', 'hb_persistence_frac',
    'ifp_tanimoto_median_vs_ref', 'lig_binding_modes_2A',
]
targets = sorted(df.target.unique())
positions = np.arange(len(targets))

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
for ax, col in zip(axes.ravel(), SHOW_FEATURES):
    if 'is_active' in df:
        data_d = [df[(df.target == t) & (df.is_active == False)][col].dropna().values for t in targets]
        data_a = [df[(df.target == t) & (df.is_active == True)][col].dropna().values for t in targets]
        bp_d = ax.boxplot(data_d, positions=positions - 0.2, widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=DECOY_C, alpha=0.55, edgecolor=NAVY),
                          medianprops=dict(color=WHITE, linewidth=1.5),
                          whiskerprops=dict(color=NAVY), capprops=dict(color=NAVY),
                          flierprops=dict(marker='o', markersize=3, markerfacecolor=NAVY, markeredgecolor=NAVY))
        bp_a = ax.boxplot(data_a, positions=positions + 0.2, widths=0.35, patch_artist=True,
                          boxprops=dict(facecolor=ACTIVE_C, alpha=0.7, edgecolor=NAVY),
                          medianprops=dict(color=NAVY, linewidth=1.5),
                          whiskerprops=dict(color=NAVY), capprops=dict(color=NAVY),
                          flierprops=dict(marker='o', markersize=3, markerfacecolor=GOLD, markeredgecolor=NAVY))
        for i, (dd, da) in enumerate(zip(data_d, data_a)):
            if len(dd):
                ax.scatter(np.full_like(dd, i - 0.2, dtype=float) + np.random.uniform(-0.07, 0.07, len(dd)),
                           dd, s=9, color=NAVY, alpha=0.5, zorder=3)
            if len(da):
                ax.scatter(np.full_like(da, i + 0.2, dtype=float) + np.random.uniform(-0.07, 0.07, len(da)),
                           da, s=12, color=GOLD, alpha=0.85, edgecolors=NAVY, linewidths=0.4, zorder=3)
    else:
        data = [df[df.target == t][col].dropna().values for t in targets]
        ax.boxplot(data, positions=positions, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor=NAVY, alpha=0.6))
    ax.set_xticks(positions); ax.set_xticklabels(targets, fontsize=9)
    ax.set_title(col, fontsize=10)
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()

**What to look for in each panel.**
- **`lig_drift_mean_A` low and `lig_escape_frac` = 0 for GOLD, higher for NAVY** — the classic "actives stay put" signal. Usually the strongest per-complex feature in VS (virtual screening) benchmarks.
- **`n_hb_mean` and `hb_persistence_frac` higher for GOLD** — persistent H-bonds are the second-strongest signal. Watch for targets where both classes sit at 0 (fully hydrophobic pocket) or both are high (large polar pocket with decoys that happen to H-bond).
- **`ifp_tanimoto_median_vs_ref` closer to 1 for GOLD** — actives preserve their initial IFP-Tanimoto (interaction-fingerprint similarity vs the reference frame); decoys drift into different contact patterns.
- **`lig_binding_modes_2A` smaller for GOLD** — fewer distinct RMSD clusters over the trajectory means the ligand picked one pose and stayed.

A feature whose actives and decoys overlap in every target isn't useful for any target. A feature that separates *some* targets can still earn a slot in an ensemble ranker with per-target weighting.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
